# 05. EfficientNet-B0 (최종 채택 모델)

timm `efficientnet_b0` (ImageNet 사전학습) + Dropout(0.3) + 6 class 헤드, Adam(lr 1e-4), StepLR, 5 epoch.
Kaggle API로 이미지 데이터셋을 직접 내려받아 Colab에서 학습했습니다.


In [ ]:
# 1) 환경변수 설정
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

In [ ]:
# 2) kaggle 설치 및 데이터셋 다운로드
!pip install kaggle -q
!kaggle datasets download -d paramaggarwal/fashion-product-images-dataset --unzip

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset
License(s): MIT
100% 23.1G/23.1G [03:44<00:00, 110MB/s]



In [ ]:
# 3) 경로 확인
import os
print(os.listdir('/content/'))

['.config', 'fashion-dataset', 'sample_data']


In [ ]:
print(os.listdir('/content/fashion-dataset/'))

['images.csv', 'fashion-dataset', 'styles.csv', 'styles', 'images']


In [ ]:
# 4) IMG_DIR 설정
IMG_DIR = '/content/fashion-dataset/images'

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

class FashionDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        self.classes = ['Casual', 'Formal', 'Romantic', 'Sporty', 'Street', 'Vintage']
        self.label2idx = {c: i for i, c in enumerate(self.classes)}
        self.df = self.df[self.df['style_label'].isin(self.classes)].reset_index(drop=True)

        # 실제로 존재하는 이미지만 필터링
        self.df = self.df[
            self.df['id'].apply(
                lambda x: os.path.exists(os.path.join(img_dir, str(x) + '.jpg'))
            )
        ].reset_index(drop=True)

        print(f"유효한 이미지 수: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, str(row['id']) + '.jpg')
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.label2idx[row['style_label']]
        return image, label

In [ ]:
from google.colab import files
files.upload()  # train (1).csv, val (1).csv 두 개 선택해서 업로드

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((240, 240)),       # EfficientNet-B0 권장 입력 크기
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])  # ImageNet 기준
])

val_transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

IMG_DIR = '/content/fashion-dataset/images'

train_dataset = FashionDataset('/content/train (1).csv', IMG_DIR, train_transform)
val_dataset   = FashionDataset('/content/val (1).csv',   IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4)

유효한 이미지 수: 21398
유효한 이미지 수: 3058


In [ ]:
import timm
import torch.nn as nn

class EfficientNetFashion(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        # ImageNet 사전학습 가중치 로드
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True)

        # 마지막 분류층을 6개 클래스로 교체 (Multi-Task Head 자리)
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

    def predict_proba(self, x):
        """향조 매핑에 넣을 softmax 확률 벡터 반환"""
        logits = self.forward(x)
        return torch.softmax(logits, dim=1)

model = EfficientNetFashion(num_classes=6)

In [ ]:
import torch.optim as optim

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS = 5

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_loss, train_correct = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()

    # ── Validation ──
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss    += criterion(outputs, labels).item()
            val_correct += (outputs.argmax(1) == labels).sum().item()

    scheduler.step()

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Train Acc: {train_correct/len(train_dataset):.3f} | "
          f"Val Acc: {val_correct/len(val_dataset):.3f}")

# 모델 저장
torch.save(model.state_dict(), 'efficientnet_fashion.pth')

Epoch  1/5 | Train Loss: 0.5040 | Train Acc: 0.837 | Val Acc: 0.899
Epoch  2/5 | Train Loss: 0.2567 | Train Acc: 0.910 | Val Acc: 0.907
Epoch  3/5 | Train Loss: 0.1910 | Train Acc: 0.932 | Val Acc: 0.910
Epoch  4/5 | Train Loss: 0.1473 | Train Acc: 0.945 | Val Acc: 0.913
Epoch  5/5 | Train Loss: 0.1146 | Train Acc: 0.959 | Val Acc: 0.916


In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/efficientnet_fashion.pth')

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

CLASSES = ['Casual', 'Formal', 'Romantic', 'Sporty', 'Street', 'Vintage']
print(classification_report(all_labels, all_preds, target_names=CLASSES))

              precision    recall  f1-score   support

      Casual       0.94      0.95      0.94      2157
      Formal       0.89      0.84      0.86       183
    Romantic       1.00      0.83      0.91        53
      Sporty       0.83      0.76      0.79       363
      Street       0.71      0.67      0.69        54
     Vintage       0.91      0.94      0.92       248

    accuracy                           0.92      3058
   macro avg       0.88      0.83      0.85      3058
weighted avg       0.91      0.92      0.91      3058



In [ ]:
from google.colab import files
files.upload()  # style_accord_mapping.json 업로드

In [ ]:
import json
import numpy as np

# 매핑 행렬 로드
with open('style_accord_mapping.json', 'r') as f:
    mapping = json.load(f)
mapping_df = pd.DataFrame(mapping).T   # (6×5)

CLASSES = ['Casual', 'Formal', 'Romantic', 'Sporty', 'Street', 'Vintage']
# [수정] JSON 키 순서(Romantic, Casual, ...)와 모델 출력 순서(CLASSES)가 달라
#        행렬곱 시 스타일이 엇갈려 매칭되던 문제 → 행 순서를 CLASSES에 맞춤
mapping_df = mapping_df.loc[CLASSES]

def recommend(image_path, model, mapping_df):
    model.eval()
    img = val_transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = model.predict_proba(img).cpu().numpy()[0]

    style_vec = pd.Series(probs, index=CLASSES)
    accord_scores = pd.Series(
        style_vec.values @ mapping_df.values,
        index=mapping_df.columns
    ).sort_values(ascending=False)

    print("스타일 확률:")
    for s, p in style_vec.sort_values(ascending=False).items():
        print(f"  {s}: {p:.3f}")
    print("향조 점수:")
    for a, s in accord_scores.items():
        print(f"  {a}: {s:.3f}")

    return accord_scores

> **수정 사항**: 원래 실행에서는 매핑 행렬의 행 순서가 모델 출력 순서와 달라 Casual(0.974)로 분류된 이미지에 Romantic 행의 향조(floral 0.346, romantic 0.264)가 계산되었습니다.
> 위 셀에서 행 순서를 맞추도록 수정했고, 이전 실행 결과는 잘못된 값이므로 아래 셀의 출력을 지웠습니다.


In [ ]:
recommend('/content/fashion-dataset/images/47889.jpg', model, mapping_df)